In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy import interpolate
from scipy.signal import resample as sci_resample
import glob
import mne
from eelbrain import *
import os
import pandas as pd
import seaborn as sns

#### Get envelope

In [ ]:
def rms_interpolate(audio_file, eeg_sr=1000, eeg_duration=[100,701]):

    stim, sr = librosa.load(audio_file)

    # Compute RMS 
    rms_win = 0.01 # 10ms
    rms_hop = 1/eeg_sr # hop by eeg sampling rate
    rms = librosa.feature.rms(y=stim,
                            frame_length=int(sr*rms_win),
                            hop_length=int(sr*rms_hop))
    rms_sr = 1/rms_hop # the rms time series is sampled with period rms_hop
    rms=rms[0]

    # resample to same rate as EEG
    rms_resampled = sci_resample(rms,num=int(eeg_sr*len(rms)/rms_sr))
    rms_resampled=rms_resampled[:eeg_duration[1]]
    
    # pad 
    pad_before = eeg_duration[0]
    pad_after = np.max([0,eeg_duration[1] - len(rms_resampled)])
    rms_padded = np.pad(rms_resampled, pad_width=(pad_before,pad_after))
    rms_padded = rms_padded.astype('<f8')
    rms_padded = np.where(np.isfinite(rms_padded), rms_padded, 0)

    return rms_padded

#### Plot envelope

In [ ]:
rms_std = rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
rms_looming = rms_interpolate('./sounds/deviant_looming_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
rms_receding = rms_interpolate('./sounds/deviant_receding_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
rms_flat = rms_interpolate('./sounds/deviant_flat_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')

times_long = np.arange(-0.100, 0.701, 0.001)

fig, ax = plt.subplots(nrows=4, sharex=True, sharey=True, figsize=(8, 8))
fig.tight_layout(pad=5)
ax[0].plot(times_long, rms_std)
ax[0].set_title('Standard RMS')
ax[1].plot(times_long, rms_looming)
ax[1].set_title('Looming RMS')
ax[2].plot(times_long, rms_receding)
ax[2].set_title('Receding RMS')
ax[3].plot(times_long, rms_flat)
ax[3].set_title('Flat RMS')

fig, ax = plt.subplots(nrows=3, sharex=True, sharey=True, figsize=(8, 8))
fig.tight_layout(pad=5)
ax[0].plot(times_long, rms_looming - rms_std)
ax[0].set_title('Looming-Standard RMS')
ax[1].plot(times_long, rms_receding - rms_std)
ax[1].set_title('Receding-Standard RMS')
ax[2].plot(times_long, rms_flat - rms_std)
ax[2].set_title('Flat-Standard RMS')

#### Create dataset (all subjects)

In [ ]:
stim_dict = {
                '1002': rms_interpolate('./sounds/deviant_looming_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav') - rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav'), 
                '1003': rms_interpolate('./sounds/deviant_receding_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav') - rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav'), 
                '1004': rms_interpolate('./sounds/deviant_flat_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav') - rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
            }

In [ ]:
subjs_all = glob.glob('./analysis_800ms_epochs/looming*-epo.fif')
del subjs_all[2]
del subjs_all[5]
print(subjs_all)
subjs = []
for i in subjs_all:
    subjs.append(i.lower())

subjs = sorted(subjs, key=lambda x: int(x.split('-')[0][-3:]))

In [ ]:
tstep = 1. / 1000
n_times = 801
time = UTS(-0.100, tstep, n_times)

sensor = Sensor.from_montage('easycap-M1')[:64]

In [ ]:
rows = []

standard_evks = []
looming_evks = []
receding_evks = []
flat_evks = []

for subj in subjs:
    subj_epochs = mne.read_epochs(subj, preload=True, verbose=False)
    subj_epochs = subj_epochs.drop_channels('STI')

    for i in range(len(subj_epochs)):
        if subj_epochs[i].events[:, 2][0] != 1001:

            subject = int(subj.split('-')[0][-3:])

            condition = str(subj_epochs[i].events[:, 2][0])

            eeg_deviant = load.fiff.epochs_ndvar(subj_epochs[i])
            eeg_std = load.fiff.epochs_ndvar(subj_epochs[i-1])
            eeg = eeg_deviant - eeg_std
            eeg.name = 'EEG'

            e = stim_dict.get(str(subj_epochs[i].events[:, 2][0]))
            envelope = NDVar(e, (time,), name='envelope')

            rows.append([subject, condition, eeg, envelope])

ds = Dataset.from_caselist(['subject','condition', 'eeg', 'envelope'], rows)
ds['subject'].random = True
print(ds.summary())

In [ ]:
ds = load.unpickle('./pickles/ds.pickle')

In [ ]:
# Separate ds into separate datasets for each condition
ds_looming = ds.sub(ds['condition'] == '1002')
ds_receding = ds.sub(ds['condition'] == '1003')
ds_flat = ds.sub(ds['condition'] == '1004')

ds_flat_receding = ds.sub(ds['condition'] != '1002')
print(ds_flat_receding.summary())

#### Compute TRF (all subjects)

In [ ]:
# Train model on stimuli difference and ERP difference waves
fit = boosting(
                'eeg',          # signal to predict (each channel is computed independently)
                'envelope',     # predictor
                0, 
                0.700, 
                basis=0.05, 
                ds=ds_flat,     # the dataset to train on
                delta=0.005, 
                partitions=2, 
                # test=1
                # selective_stopping=True
            )

In [ ]:
# Plot TRF
p = plot.TopoButterfly(fit.h_scaled, w=6, h=2)
p.set_time(.380)
p.plot_colorbar();

#### Predict using TRF

In [ ]:
fit_looming = load.unpickle('./pickles/trf_looming.pickle')
fit_receding = load.unpickle('./pickles/trf_receding.pickle')
fit_flat = load.unpickle('./pickles/trf_flat.pickle')

In [ ]:
diff_looming_ndvar = ds_looming['eeg'].mean('case')
diff_looming_ndvar.name = 'looming'
diff_receding_ndvar = ds_receding['eeg'].mean('case')
diff_receding_ndvar.name = 'receding'
diff_flat_ndvar = ds_flat['eeg'].mean('case')
diff_flat_ndvar.name = 'flat'

colors_erp = {
    ('looming'): plot.Style('r', linestyle='-', linewidth=1.5),
    ('receding'): plot.Style('k', linestyle='-', linewidth=1.5),
    ('flat'): plot.Style('b', linestyle='-', linewidth=1.5),
}

# Plot difference waves in Fz for each deviant condition
p = plot.UTS([[diff_looming_ndvar.sub(sensor='Fz'), diff_receding_ndvar.sub(sensor='Fz'), diff_flat_ndvar.sub(sensor='Fz')]], colors=colors_erp);
p.set_ylim([3e-6, -3e-6])
p.add_hline(y=0, ls='--', c='k')
p.add_vline(x=0, ls='--', c='k')
p.plot_legend(loc='upper left')

In [ ]:
# x is the stimulus from which to predict the EEG difference wave
x_looming = NDVar(rms_looming - rms_std, (time,))
x_receding = NDVar(rms_receding - rms_std, (time,))
x_flat = NDVar(rms_flat - rms_std, (time,))

# Convolve TRF with stimulus (x) to predict the EEG difference wave
y_pred_looming = convolve(fit_looming.h_scaled, x_looming, name='looming prediction')
y_pred_receding = convolve(fit_receding.h_scaled, x_receding, name='receding prediction')
y_pred_flat = convolve(fit_flat.h_scaled, x_flat, name='flat prediction')


colors = {
    ('looming'): plot.Style('r', linestyle='-', linewidth=0.75),
    ('receding'): plot.Style('k', linestyle='-', linewidth=0.75),
    ('flat'): plot.Style('b', linestyle='-', linewidth=0.75),
    ('looming prediction'): plot.Style('r', linestyle='--', linewidth=0.75),
    ('receding prediction'): plot.Style('k', linestyle='--', linewidth=0.75),
    ('flat prediction'): plot.Style('b', linestyle='--', linewidth=0.75),
}

p = plot.UTS([
                [y_pred_looming.sub(sensor='Fz'), diff_looming_ndvar.sub(sensor='Fz')], 
                [y_pred_receding.sub(sensor='Fz'), diff_receding_ndvar.sub(sensor='Fz')], 
                [y_pred_flat.sub(sensor='Fz'), diff_flat_ndvar.sub(sensor='Fz')]
             ],
                colors=colors,
                # legend='lower right',
            )
p.set_ylim([3e-6, -3e-6])

#### Compute TRFs (each subject)

In [ ]:
actuals = []
trfs = []
predictions = []

for subj in subjs:
    subject = int(subj.split('-')[0][-3:])
    ds_subj = ds_flat_receding.sub(ds_flat_receding['subject'] == subject)
    ds_comp = ds_looming.sub(ds_looming['subject'] == subject)

    tstep = 1. / 1000
    n_times = 801
    time = UTS(-0.100, tstep, n_times)

    actuals.append(ds_comp['eeg'].mean('case').sub(sensor='Fz'))

    # Train model on stimuli difference and ERP difference waves
    fit = boosting(
                    'eeg',          # signal to predict (each channel is computed independently)
                    'envelope',     # predictor
                    0, 
                    0.700, 
                    # basis=1, 
                    # basis_window='exponential',
                    ds=ds_subj,     # the dataset to train on
                    # delta=0.001, 
                    partitions=2, 
                    # test=1
                    # selective_stopping=True
                )
    trfs.append(fit.h_scaled)
    predictions.append(convolve(fit.h_scaled, NDVar(rms_looming - rms_std, (time,)), name='prediction').sub(sensor='Fz'))

In [ ]:
ds = Dataset()

subjects = [int(subj.split('-')[0][-3:]) for subj in subjs]
subjects_rep = np.repeat(subjects, 801*2)

conditions = ['actual', 'predicted']
conditions_rep = np.tile(np.repeat(conditions, 801), 15)

time = np.arange(-0.100, 0.701, 0.001)
time_rep = np.tile(time, 30)

final_eeg = []
for i in range(len(actuals)):
    for j in range(len(actuals[0])):
        final_eeg.append(actuals[i].x[j])
    for k in range(len(predictions[0])):
        final_eeg.append(predictions[i].x[k])

ds['subject'] = Var(subjects_rep)
ds['condition'] = Factor(conditions_rep)
ds['time'] = time_rep
ds['Fz'] = Var(final_eeg)

In [ ]:
print(ds.summary())

In [ ]:
df = ds.as_dataframe()
df.head()

In [ ]:
ax = sns.lineplot(
                df, 
                x='time', 
                y='Fz', 
                hue='condition', 
                # errorbar=('ci', 68)
            )
ax.set_title('Train: Flat-Std + Receding-Std, Predict: Looming-Std')
# ax.set_ylim([4e-6, -4e-6])
ax.invert_yaxis()

#### Backward model (each subject)

In [ ]:
actuals = []
# trfs = []
predictions = []
subj_epochs = mne.read_epochs('./analysis_800ms_epochs/Looming005-epo.fif', preload=True, verbose=False)
subj_epochs = subj_epochs.drop_channels('STI')
test_eeg = load.fiff.epochs_ndvar(subj_epochs['1004'])
test_eeg = test_eeg.mean('case')

for subj in subjs:
    subject = int(subj.split('-')[0][-3:])
    ds_subj = ds_flat.sub(ds_flat['subject'] == subject)

    tstep = 1. / 1000
    n_times = 801
    time = UTS(-0.100, tstep, n_times)

    actuals.append(rms_flat - rms_std)

    # Train model on stimuli difference and ERP difference waves
    fit = boosting(
                    'envelope',          # signal to predict (each channel is computed independently)
                    'eeg',     # predictor
                    0, 
                    0.700, 
                    # basis=1, 
                    # basis_window='exponential',
                    ds=ds_subj,     # the dataset to train on
                    # delta=0.001, 
                    partitions=2, 
                    # test=1
                    # selective_stopping=True
                )
    # trfs.append(fit.h_scaled)
    predictions.append(convolve(fit.h_scaled, test_eeg, name='prediction'))

In [ ]:
ds = Dataset()

subjects = [int(subj.split('-')[0][-3:]) for subj in subjs]
subjects_rep = np.repeat(subjects, 801*2)

conditions = ['actual', 'predicted']
conditions_rep = np.tile(np.repeat(conditions, 801), 15)

time = np.arange(-0.100, 0.701, 0.001)
time_rep = np.tile(time, 30)

stim = []
for i in range(len(actuals)):
    for j in range(len(actuals[0])):
        stim.append(actuals[i][j])
    for k in range(len(predictions[0])):
        stim.append(predictions[i].x[k])

ds['subject'] = Var(subjects_rep)
ds['condition'] = Factor(conditions_rep)
ds['time'] = time_rep
ds['stim'] = Var(stim)

In [ ]:
print(ds.summary())

In [ ]:
df = ds.as_dataframe()
df.head()

In [ ]:
df.loc[df['condition'] == 'predicted']['stim'] *= 100

In [ ]:
ax = sns.lineplot(
                df, 
                x='time', 
                y=df['stim'], 
                hue='condition', 
                errorbar=('ci', 68)
            )
ax.set_title('Train: Flat-Std, Predict: Flat-Std')